# ASL data collection - **WORSE**

Records short webcam takes of the sign **WORSE** for the new training set. Every take you approve is
saved as a video clip plus a row in `manifest.csv`. Keypoints are extracted later from the clips, so there is
**nothing to install** in this notebook.

**How to use (desktop Chrome, laptop webcam):**

1. In **1. Config**, type your first name in `PERSON` (lowercase) and press play.
2. Press play on **Setup** (it mounts Google Drive so takes survive a disconnect - click Allow).
3. Press play on **2. Record takes**. Allow camera access when Chrome asks.
4. For each take the word flashes, a 3-2-1 countdown runs, then it records
   90 frames (3.0 s) with a frame counter in the corner. Sign **WORSE** once during the take.
5. Replay the take (pause, slow-mo, scrub), then click **Good - save** or **Bad - retry**.
6. It stops after **30** good takes. Re-running the cell resumes where you left off.
7. Run **3. Summary** at the end to check the results (and optionally download a zip).

**Tips:** keep your head and both hands in frame the whole time, sit about where the glasses camera would see a
signer, use even lighting, and use the same camera position for every take. Saved frames are never mirrored.
Do not change `FPS`, `CLIP_SECONDS`, `WIDTH` or `HEIGHT` - they must match across the whole team.

**If you see an error:**

- *"Set PERSON in the config cell"*: type your first name in `PERSON` and re-run the Config cell.
- *No camera preview*: use desktop Chrome, click Allow on the camera prompt, and close other apps using the webcam.
- *"Capture stuttered"* warning: close other apps using the camera and retry the take.
- After any restart, re-run Config, Setup, then Record.

In [ ]:
# ---- 1. Config ------------------------------------------------------------
# Only PERSON (and optionally TAKES) should be edited. Everything else must stay
# identical for every teammate so the datasets line up.

WORD = 'worse'                  # label / folder name - do not change
WORD_DISPLAY = 'WORSE'       # text flashed on screen

# Your first name, lowercase letters/digits, e.g. "newton"
PERSON = "" #@param {type:"string"}
# Good takes to collect for this word
TAKES = 30 #@param {type:"integer"}

FPS = 30                   # capture rate; 30 lets you derive 15 or 10 fps later
CLIP_SECONDS = 3.0         # 3.0 s -> 90 frames
WIDTH, HEIGHT = 640, 360   # 16:9, matches the glasses stream aspect ratio
JPEG_QUALITY = 0.85
COUNTDOWN_SECONDS = 3
AUTO_CONTINUE = True       # False = click "Start next take" between takes
MIRROR_PREVIEW = False     # cosmetic: mirrors only the live preview, never the saved frames

USE_DRIVE = True           # save to Google Drive (recommended; survives disconnects)
DRIVE_FOLDER = "ASL_collection"
LOCAL_FOLDER = "/content/asl_collection_output"   # used if Drive is off/unavailable

In [ ]:
#@title Setup - press play (do not edit) { display-mode: "form" }
# Python side of the Colab ASL data collection notebook (video only).
#
# make_notebooks.py inlines this file verbatim into the hidden "Setup" cell.
# It expects these names to already exist (defined in the visible config cell):
#   WORD, WORD_DISPLAY, PERSON, TAKES, FPS, CLIP_SECONDS, WIDTH, HEIGHT,
#   JPEG_QUALITY, COUNTDOWN_SECONDS, AUTO_CONTINUE, MIRROR_PREVIEW,
#   USE_DRIVE, DRIVE_FOLDER, LOCAL_FOLDER
#
# What it does:
#   - resolves the output folder (Google Drive if possible, local fallback)
#   - receives the frames of each take from the browser and checks the capture
#     timing (stutter / repeated frames)
#   - saves  <word>/<word>_<person>_<NNN>.mp4  and appends a row to manifest.csv
#   - exposes the callbacks the browser UI (app.js) calls via google.colab.output
#
# No MediaPipe here on purpose: keypoints are extracted later, from these videos,
# with whichever extractor the model is trained on. Only OpenCV and NumPy are
# needed, and both are preinstalled in Colab.

import base64
import csv
import datetime
import json
import os
import re
import shutil
import tempfile
import traceback
import zipfile

import cv2
import numpy as np

try:
    from IPython.display import JSON as _JSON
except Exception:  # pragma: no cover - only hit outside IPython
    _JSON = None


def _slug(text):
    return re.sub(r"[^a-z0-9]+", "_", str(text).strip().lower()).strip("_")


WORD_ID = _slug(WORD)
PERSON_ID = _slug(PERSON)
if not WORD_ID:
    raise ValueError("WORD is empty - do not edit it; it is the label/folder name.")
if not PERSON_ID:
    raise ValueError("Set PERSON in the config cell (your first name, e.g. 'newton') and re-run.")

NUM_FRAMES = int(round(FPS * CLIP_SECONDS))

# ----------------------------------------------------------------- output folder
ON_DRIVE = False


def _resolve_output_root():
    global ON_DRIVE
    if USE_DRIVE:
        try:
            from google.colab import drive  # type: ignore

            drive.mount("/content/drive")
            root = os.path.join("/content/drive/MyDrive", DRIVE_FOLDER)
            os.makedirs(root, exist_ok=True)
            ON_DRIVE = True
            return root
        except Exception as exc:
            print(f"[setup] Google Drive not available ({exc!r}); saving locally instead.")
    root = os.path.abspath(LOCAL_FOLDER)
    os.makedirs(root, exist_ok=True)
    return root


OUT_ROOT = _resolve_output_root()
WORD_DIR = os.path.join(OUT_ROOT, WORD_ID)
os.makedirs(WORD_DIR, exist_ok=True)
MANIFEST_PATH = os.path.join(OUT_ROOT, "manifest.csv")
TMP_DIR = tempfile.mkdtemp(prefix="asl_collect_")

MANIFEST_COLUMNS = [
    "word", "person", "take", "file_base", "clip_file", "fps", "num_frames", "width", "height",
    "camera_fps", "measured_fps", "max_gap_ms", "dup_frames", "jpeg_quality", "saved_utc",
]


# -------------------------------------------------------------------- capture QA
def build_qa(n_frames, meta):
    """Capture-timing check only. Hands/pose are checked later, when keypoints are extracted."""
    n = max(int(n_frames), 1)
    ts = [float(t) for t in (meta.get("ts") or [])]
    if len(ts) >= 2 and ts[-1] > ts[0]:
        measured_fps = (len(ts) - 1) / ((ts[-1] - ts[0]) / 1000.0)
        max_gap = max(b - a for a, b in zip(ts[:-1], ts[1:]))
    else:
        measured_fps, max_gap = 0.0, 0.0
    dups = int(meta.get("dup_frames", -1))
    stuttered = (measured_fps < 0.85 * FPS) or (dups > 0.10 * n) or (max_gap > 3.0 * 1000.0 / FPS)
    return {
        "frames": int(n_frames),
        "measured_fps": float(measured_fps),
        "max_gap_ms": float(max_gap),
        "stuttered": bool(stuttered),
    }


# ------------------------------------------------------------------- saving
def _decode_jpeg_b64(b64):
    arr = np.frombuffer(base64.b64decode(b64), dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("could not decode a JPEG frame from the browser")
    if img.shape[1] != WIDTH or img.shape[0] != HEIGHT:
        img = cv2.resize(img, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)
    return img


def _write_clip(frames_bgr, base):
    """Write the downscaled clip locally. mp4 first, MJPG .avi as a safety net."""
    path = os.path.join(TMP_DIR, base + ".mp4")
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), float(FPS), (WIDTH, HEIGHT))
    if not writer.isOpened():
        writer.release()
        path = os.path.join(TMP_DIR, base + ".avi")
        writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"MJPG"), float(FPS), (WIDTH, HEIGHT))
        if not writer.isOpened():
            writer.release()
            raise RuntimeError("OpenCV could not open a video writer (mp4v or MJPG).")
    for frame in frames_bgr:
        writer.write(frame)
    writer.release()
    return path


def _verify_clip(path, expected_frames):
    """Re-open the written clip so an unreadable or truncated file is never saved as 'good'."""
    cap = cv2.VideoCapture(path)
    try:
        count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        ok, _ = cap.read()
    finally:
        cap.release()
    if not ok or abs(count - expected_frames) > 1:
        raise RuntimeError(
            "the written clip looks wrong (%d frames, expected %d); take not saved" % (count, expected_frames)
        )


_TAKE_RE = re.compile(r"^%s_%s_(\d+)\.(?:mp4|avi)$" % (re.escape(WORD_ID), re.escape(PERSON_ID)))


def existing_take_indices():
    found = []
    for name in os.listdir(WORD_DIR):
        m = _TAKE_RE.match(name)
        if m:
            found.append(int(m.group(1)))
    return sorted(found)


def _append_manifest(row):
    """Append one row. If manifest.csv already has a header (e.g. from an older notebook
    version with extra columns), keep that header and fill what we have."""
    fieldnames = MANIFEST_COLUMNS
    new_file = not os.path.exists(MANIFEST_PATH) or os.path.getsize(MANIFEST_PATH) == 0
    if not new_file:
        with open(MANIFEST_PATH, newline="", encoding="utf-8") as fh:
            header = next(csv.reader(fh), None)
        if header:
            fieldnames = header
    with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore", restval="")
        if new_file:
            writer.writeheader()
        writer.writerow(row)


def save_current_take():
    """Persist the checked take. Order: clip, then manifest row (manifest = completed takes)."""
    s = SESSION
    if not s["bgr"] or s["qa"] is None:
        raise RuntimeError("no checked take to save")
    indices = existing_take_indices()
    idx = (indices[-1] + 1) if indices else 1
    base = "%s_%s_%03d" % (WORD_ID, PERSON_ID, idx)

    clip_tmp = _write_clip(s["bgr"], base)
    _verify_clip(clip_tmp, len(s["bgr"]))
    clip_name = os.path.basename(clip_tmp)
    shutil.copyfile(clip_tmp, os.path.join(WORD_DIR, clip_name))
    os.remove(clip_tmp)

    qa, meta = s["qa"], s["meta"]
    _append_manifest(
        {
            "word": WORD_ID,
            "person": PERSON_ID,
            "take": idx,
            "file_base": base,
            "clip_file": clip_name,
            "fps": FPS,
            "num_frames": len(s["bgr"]),
            "width": WIDTH,
            "height": HEIGHT,
            "camera_fps": meta.get("camera_fps") or "",
            "measured_fps": round(qa["measured_fps"], 2),
            "max_gap_ms": round(qa["max_gap_ms"], 1),
            "dup_frames": meta.get("dup_frames", -1),
            "jpeg_quality": JPEG_QUALITY,
            "saved_utc": datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        }
    )
    return idx, base


# ---------------------------------------------------------- browser callbacks
SESSION = {"take_id": None, "bgr": [], "qa": None, "meta": None}


def _reset_session(take_id=None):
    SESSION.update({"take_id": take_id, "bgr": [], "qa": None, "meta": None})


def _reply(obj):
    return _JSON(obj) if _JSON is not None else obj


def _safe(fn):
    """Return {'error': ...} to the browser instead of raising (so the UI can show it)."""

    def wrapper(*args):
        try:
            return _reply(fn(*args))
        except Exception as exc:  # noqa: BLE001
            traceback.print_exc()
            return _reply({"error": "%s: %s" % (type(exc).__name__, exc)})

    wrapper.__name__ = fn.__name__
    return wrapper


def _check_take(take_id):
    if SESSION["take_id"] != take_id:
        raise RuntimeError("take id mismatch (browser %r vs python %r) - re-run the cell" % (take_id, SESSION["take_id"]))


@_safe
def cb_init():
    _reset_session()
    return {
        "ok": True,
        "saved": len(existing_take_indices()),
        "out_dir": WORD_DIR,
        "on_drive": ON_DRIVE,
    }


@_safe
def cb_begin(take_id):
    _reset_session(take_id)
    return {"ok": True}


@_safe
def cb_upload(take_id, start, *chunks):
    _check_take(take_id)
    if int(start) != len(SESSION["bgr"]):
        raise RuntimeError("frame upload out of order (expected %d, got %s)" % (len(SESSION["bgr"]), start))
    for b64 in chunks:
        SESSION["bgr"].append(_decode_jpeg_b64(b64))
    return {"ok": True, "n": len(SESSION["bgr"])}


@_safe
def cb_check(take_id, meta_json):
    _check_take(take_id)
    frames = SESSION["bgr"]
    if len(frames) != NUM_FRAMES:
        raise RuntimeError("expected %d frames, got %d" % (NUM_FRAMES, len(frames)))
    meta = json.loads(meta_json) if meta_json else {}
    qa = build_qa(len(frames), meta)
    SESSION.update({"qa": qa, "meta": meta})
    return {"ok": True, "qa": qa}


@_safe
def cb_save(take_id):
    _check_take(take_id)
    idx, base = save_current_take()
    _reset_session()
    return {"ok": True, "saved": len(existing_take_indices()), "take": idx, "file": base}


@_safe
def cb_discard(take_id):
    _reset_session()
    return {"ok": True}


@_safe
def cb_finish():
    _reset_session()
    return {"ok": True, "saved": len(existing_take_indices()), "out_dir": WORD_DIR}


CALLBACKS = {
    "asl.init": cb_init,
    "asl.begin": cb_begin,
    "asl.upload": cb_upload,
    "asl.check": cb_check,
    "asl.save": cb_save,
    "asl.discard": cb_discard,
    "asl.finish": cb_finish,
}


def register_callbacks(output_module=None):
    """Register the Python callbacks that the browser UI invokes."""
    if output_module is None:
        from google.colab import output as output_module  # type: ignore
    for name, fn in CALLBACKS.items():
        output_module.register_callback(name, fn)


def ui_config():
    """Settings handed to app.js (replaces the __CFG__ token)."""
    return {
        "word": WORD_ID,
        "wordDisplay": WORD_DISPLAY,
        "person": PERSON_ID,
        "fps": int(FPS),
        "numFrames": int(NUM_FRAMES),
        "width": int(WIDTH),
        "height": int(HEIGHT),
        "jpegQuality": float(JPEG_QUALITY),
        "countdown": int(COUNTDOWN_SECONDS),
        "target": int(TAKES),
        "autoContinue": bool(AUTO_CONTINUE),
        "mirrorPreview": bool(MIRROR_PREVIEW),
        "flashMs": 1500,
        "stepMs": 1000,
    }


# ------------------------------------------------------------ after the session
def summarize():
    """Print what was saved for this word/person (reads manifest.csv)."""
    rows = []
    if os.path.exists(MANIFEST_PATH):
        with open(MANIFEST_PATH, newline="", encoding="utf-8") as fh:
            rows = [r for r in csv.DictReader(fh) if r.get("word") == WORD_ID and r.get("person") == PERSON_ID]
    print("Word: %s   Person: %s" % (WORD_ID, PERSON_ID))
    print("Saved takes: %d (target %d)" % (len(rows), TAKES))
    print("Folder: %s%s" % (WORD_DIR, "   [Google Drive]" if ON_DRIVE else "   [local Colab disk - download the zip!]"))
    if rows:
        fps = [float(r["measured_fps"]) for r in rows if r.get("measured_fps")]
        if fps:
            print("Measured capture fps: mean %.1f, min %.1f" % (np.mean(fps), min(fps)))
            slow = [r["file_base"] for r in rows if r.get("measured_fps") and float(r["measured_fps"]) < 0.85 * FPS]
            if slow:
                print("Low capture fps, consider re-recording:", ", ".join(slow))
    return rows


def make_zip():
    """Zip this word's folder plus the manifest and offer it as a download (Colab)."""
    zip_path = os.path.join(tempfile.gettempdir(), "asl_%s_%s.zip" % (WORD_ID, PERSON_ID))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for name in sorted(os.listdir(WORD_DIR)):
            zf.write(os.path.join(WORD_DIR, name), os.path.join(WORD_ID, name))
        if os.path.exists(MANIFEST_PATH):
            zf.write(MANIFEST_PATH, "manifest.csv")
    print("Wrote", zip_path)
    try:
        from google.colab import files  # type: ignore

        files.download(zip_path)
    except Exception:
        print("(Not in Colab - copy the zip from the path above.)")
    return zip_path


print(
    "[setup] word=%s person=%s | %d frames @ %d fps (%.1fs) at %dx%d"
    % (WORD_ID, PERSON_ID, NUM_FRAMES, FPS, CLIP_SECONDS, WIDTH, HEIGHT)
)
print("[setup] output: %s%s | already saved: %d" % (WORD_DIR, " (Google Drive)" if ON_DRIVE else " (local)", len(existing_take_indices())))

In [ ]:
#@title 2. Record takes - press play { display-mode: "form" }
import json
from IPython.display import display, Javascript

APP_JS = r'''
// Browser-side collector UI for the Colab ASL data collection notebooks.
//
// This file is inlined by make_notebooks.py into the "Run collection" cell and
// executed with IPython.display.Javascript. The config placeholder on the
// "const CFG" line below is replaced with a JSON object built by
// helpers.py:ui_config(). (The placeholder must appear exactly once in this file.)
//
// Flow per take:
//   flash word -> countdown 3..1 -> record N frames (corner counter) ->
//   upload frames to Python -> capture-timing check -> review/replay -> Good (save) / Bad (retry)
//
// Frames are always saved UNMIRRORED. MIRROR_PREVIEW only changes the live preview.
(async () => {
  const CFG = __CFG__;
  const FPS = CFG.fps;
  const N = CFG.numFrames;
  const W = CFG.width;
  const H = CFG.height;
  const sleep = (ms) => new Promise((r) => setTimeout(r, ms));
  const S = { stop: false };
  const ui = { mode: 'live', banner: null, count: null, rec: null, slate: null };
  let replay = null;
  // ---------------------------------------------------------------- DOM
  const root = document.createElement('div');
  root.style.cssText =
    'font-family:system-ui,Segoe UI,Roboto,sans-serif;max-width:780px;background:#16181d;' +
    'color:#e8eaed;padding:12px;border-radius:10px;box-sizing:border-box;';
  root.innerHTML = `
    <style>
      .aslb{padding:8px 14px;border:0;border-radius:6px;font-size:14px;cursor:pointer;background:#3a3f4b;color:#fff}
      .aslb:hover{filter:brightness(1.15)}
      .aslb.good{background:#1e8e3e;font-weight:600}
      .aslb.bad{background:#d93025;font-weight:600}
      .aslb.on{outline:2px solid #8ab4f8}
      .aslb:disabled{opacity:.5;cursor:default}
      .asl-row{display:flex;gap:8px;align-items:center;flex-wrap:wrap;margin-bottom:8px}
    </style>
    <div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px;">
      <div id="hdrWord" style="font-size:20px;font-weight:700;"></div>
      <div id="hdrCount" style="font-size:14px;opacity:.85;"></div>
    </div>
    <canvas id="view" width="${W}" height="${H}"
            style="width:100%;background:#000;border-radius:6px;display:block;"></canvas>
    <div id="msg" style="min-height:22px;margin:8px 0;font-size:14px;"></div>
    <div id="live" class="asl-row"></div>
    <div id="review" style="display:none;">
      <div id="qa" style="font-size:13px;line-height:1.5;margin-bottom:8px;"></div>
      <div class="asl-row">
        <button class="aslb" id="bPlay">Pause</button>
        <button class="aslb" id="bRestart">Replay</button>
        <span style="font-size:13px;">Speed:</span>
        <button class="aslb spd" data-s="1">1x</button>
        <button class="aslb spd" data-s="0.5">0.5x</button>
        <button class="aslb spd" data-s="0.25">0.25x</button>
      </div>
      <input type="range" id="scrub" min="0" max="${N - 1}" value="0" style="width:100%;">
      <div class="asl-row" style="margin-top:10px;">
        <button class="aslb good" id="bGood">Good - save</button>
        <button class="aslb bad" id="bBad">Bad - retry</button>
        <button class="aslb" id="bStop">Stop session</button>
      </div>
    </div>`;
  document.body.appendChild(root);
  const $ = (id) => root.querySelector('#' + id);
  const view = $('view');
  const vctx = view.getContext('2d');

  const say = (text, color) => {
    $('msg').textContent = text;
    $('msg').style.color = color || '#e8eaed';
  };
  const fail = (err) => {
    const m = err && err.message ? err.message : String(err);
    say('Error: ' + m + '  (re-run this cell to restart; saved takes are safe)', '#f28b82');
    $('live').innerHTML = '';
    $('review').style.display = 'none';
  };

  // ------------------------------------------------- Python bridge
  async function call(name, args) {
    if (!(window.google && google.colab && google.colab.kernel)) {
      throw new Error('google.colab.kernel not found - this notebook must run inside Google Colab.');
    }
    const r = await google.colab.kernel.invokeFunction(name, args || [], {});
    const out = r && r.data && r.data['application/json'];
    if (!out) throw new Error('No response from Python for ' + name);
    if (out.error) throw new Error(out.error);
    return out;
  }

  // ------------------------------------------------------ camera
  let stream = null;
  const video = document.createElement('video');
  video.muted = true;
  video.playsInline = true;
  const cap = document.createElement('canvas');
  cap.width = W;
  cap.height = H;
  const cctx = cap.getContext('2d');
  let presented = -1;
  let presentedSupported = false;
  let camFps = null;

  // Draw the current camera frame into the W x H capture canvas, center-cropped
  // to the target aspect ratio (never stretched) so geometry stays undistorted.
  function drawCap() {
    const vw = video.videoWidth;
    const vh = video.videoHeight;
    if (!vw || !vh) return;
    const targetAR = W / H;
    const srcAR = vw / vh;
    let sx = 0, sy = 0, sw = vw, sh = vh;
    if (srcAR > targetAR) {
      sw = vh * targetAR;
      sx = (vw - sw) / 2;
    } else {
      sh = vw / targetAR;
      sy = (vh - sh) / 2;
    }
    cctx.drawImage(video, sx, sy, sw, sh, 0, 0, W, H);
  }

  async function startCamera() {
    stream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 1280 }, height: { ideal: 720 }, frameRate: { ideal: 30 } },
      audio: false,
    });
    video.srcObject = stream;
    await video.play();
    for (let i = 0; i < 100 && !video.videoWidth; i++) await sleep(50);
    if (!video.videoWidth) throw new Error('Camera started but returned no video frames.');
    const track = stream.getVideoTracks()[0];
    const st = track && track.getSettings ? track.getSettings() : {};
    camFps = st.frameRate || null;
    if ('requestVideoFrameCallback' in video) {
      presentedSupported = true;
      const cb = (_now, md) => {
        presented = md.presentedFrames;
        video.requestVideoFrameCallback(cb);
      };
      video.requestVideoFrameCallback(cb);
    }
  }

  function stopCamera() {
    if (stream) stream.getTracks().forEach((t) => t.stop());
    stream = null;
  }

  // ----------------------------------------------------- drawing
  function fitFont(ctx, text, maxW, startPx, weight) {
    let px = startPx;
    do {
      ctx.font = `${weight} ${px}px system-ui,Segoe UI,Roboto,sans-serif`;
      if (ctx.measureText(text).width <= maxW) break;
      px -= 4;
    } while (px > 16);
    return px;
  }

  function centerText(text, y, startPx, color, weight) {
    fitFont(vctx, text, W * 0.9, startPx, weight || 'bold');
    vctx.fillStyle = color;
    vctx.textAlign = 'center';
    vctx.textBaseline = 'middle';
    vctx.fillText(text, W / 2, y);
  }

  let lastT = performance.now();
  function render() {
    requestAnimationFrame(render);
    const now = performance.now();
    const dt = now - lastT;
    lastT = now;

    if (ui.mode === 'slate') {
      vctx.fillStyle = '#000';
      vctx.fillRect(0, 0, W, H);
      centerText(ui.slate || 'Done', H / 2, 36, '#e8eaed', 'bold');
      return;
    }

    if (ui.mode === 'replay' && replay) {
      const n = replay.imgs.length;
      if (!replay.paused) {
        replay.pos += (dt / 1000) * FPS * replay.speed;
        if (replay.pos >= n) replay.pos -= n;
        $('scrub').value = String(Math.floor(replay.pos));
      }
      const i = Math.max(0, Math.min(n - 1, Math.floor(replay.pos)));
      vctx.drawImage(replay.imgs[i], 0, 0, W, H);
      vctx.fillStyle = 'rgba(0,0,0,0.55)';
      vctx.fillRect(W - 150, 8, 142, 26);
      vctx.fillStyle = '#e8eaed';
      vctx.font = '15px system-ui,Segoe UI,Roboto,sans-serif';
      vctx.textAlign = 'right';
      vctx.textBaseline = 'middle';
      vctx.fillText(`replay ${i + 1}/${n}  ${replay.speed}x`, W - 14, 21);
      return;
    }

    // live preview (also used during flash / countdown / recording)
    drawCap();
    if (CFG.mirrorPreview) {
      vctx.save();
      vctx.translate(W, 0);
      vctx.scale(-1, 1);
      vctx.drawImage(cap, 0, 0, W, H);
      vctx.restore();
    } else {
      vctx.drawImage(cap, 0, 0, W, H);
    }

    // persistent word label (top-left)
    vctx.fillStyle = 'rgba(0,0,0,0.55)';
    vctx.fillRect(8, 8, 220, 30);
    vctx.fillStyle = '#e8eaed';
    vctx.font = 'bold 18px system-ui,Segoe UI,Roboto,sans-serif';
    vctx.textAlign = 'left';
    vctx.textBaseline = 'middle';
    vctx.fillText(CFG.wordDisplay, 16, 24, 204);

    if (ui.banner) {
      vctx.fillStyle = 'rgba(0,0,0,0.6)';
      vctx.fillRect(0, 0, W, H);
      centerText(ui.banner.text, H * 0.42, 96, '#ffffff', 'bold');
      if (ui.banner.sub) centerText(ui.banner.sub, H * 0.72, 26, '#bdc1c6', 'normal');
    }
    if (ui.count !== null) {
      vctx.fillStyle = 'rgba(0,0,0,0.35)';
      vctx.fillRect(0, 0, W, H);
      centerText(String(ui.count), H * 0.5, 180, '#ffffff', 'bold');
    }
    if (ui.rec) {
      // red dot + frame counter in the top-right corner, progress bar at the bottom
      vctx.fillStyle = 'rgba(0,0,0,0.6)';
      vctx.fillRect(W - 190, 8, 182, 34);
      vctx.fillStyle = '#ea4335';
      vctx.beginPath();
      vctx.arc(W - 172, 25, 8, 0, 6.2832);
      vctx.fill();
      vctx.fillStyle = '#ffffff';
      vctx.font = 'bold 20px system-ui,Segoe UI,Roboto,sans-serif';
      vctx.textAlign = 'right';
      vctx.textBaseline = 'middle';
      vctx.fillText(`REC ${ui.rec.n}/${N}`, W - 16, 26);
      vctx.fillStyle = '#ea4335';
      vctx.fillRect(0, H - 6, (W * ui.rec.n) / N, 6);
    }
  }

  // ----------------------------------------------------- phases
  let note = ''; // one-shot confirmation shown on the next take's banner ("Saved ..." / "Discarded ...")

  async function flashAndCountdown(takeNo, saved) {
    ui.banner = { text: CFG.wordDisplay, sub: `${note}take ${takeNo} of ${CFG.target}  -  get ready` };
    say(note + 'Get ready to sign: ' + CFG.wordDisplay, note.startsWith('Saved') ? '#81c995' : undefined);
    note = '';
    await sleep(CFG.flashMs);
    ui.banner = null;
    for (let c = CFG.countdown; c >= 1; c--) {
      if (S.stop) break;
      ui.count = c;
      await sleep(CFG.stepMs);
    }
    ui.count = null;
  }

  async function recordClip() {
    const frames = new Array(N);
    const stamps = [];
    const pres = [];
    ui.rec = { n: 0 };
    say('Recording - sign now!');
    const t0 = performance.now();
    for (let i = 0; i < N; i++) {
      if (S.stop) {
        ui.rec = null;
        return null;
      }
      const wait = t0 + (i * 1000) / FPS - performance.now();
      if (wait > 0) await sleep(wait);
      drawCap();
      frames[i] = cap.toDataURL('image/jpeg', CFG.jpegQuality);
      stamps.push(performance.now() - t0);
      pres.push(presented);
      ui.rec.n = i + 1;
    }
    await sleep(150); // let the final N/N counter be seen before the overlay clears
    ui.rec = null;
    let dups = -1;
    if (presentedSupported) {
      dups = 0;
      for (let i = 1; i < pres.length; i++) if (pres[i] === pres[i - 1]) dups++;
    }
    return {
      frames,
      meta: {
        ts: stamps.map((v) => Math.round(v * 10) / 10),
        dup_frames: dups,
        camera_fps: camFps,
        camera_w: video.videoWidth,
        camera_h: video.videoHeight,
      },
    };
  }

  async function sendAndCheck(takeId, frames, meta) {
    await call('asl.begin', [takeId]);
    const CH = 6;
    for (let s = 0; s < frames.length; s += CH) {
      const chunk = frames.slice(s, s + CH).map((u) => u.slice(u.indexOf(',') + 1));
      say(`Uploading frames ${Math.min(s + CH, frames.length)}/${frames.length} ...`);
      await call('asl.upload', [takeId, s, ...chunk]);
    }
    say('Checking the capture ...');
    return await call('asl.check', [takeId, JSON.stringify(meta)]);
  }

  function describeQA(qa, meta) {
    const lines = [];
    lines.push(
      'Watch the replay: your head and <b>both hands</b> should stay in frame for the whole sign, ' +
        'and you should sign once. If not, click Bad - retry.'
    );
    const bits = [`${qa.measured_fps.toFixed(1)} fps measured`];
    if (meta.dup_frames >= 0) bits.push(`${meta.dup_frames} repeated frames`);
    if (meta.camera_fps) bits.push(`camera reports ${Math.round(meta.camera_fps)} fps`);
    bits.push(`max gap ${Math.round(qa.max_gap_ms)} ms`);
    lines.push(`<b>Capture</b> - ${bits.join(' | ')}`);
    const warn = [];
    if (qa.stuttered) {
      warn.push('Capture stuttered (low fps or many repeated frames) - check lighting / close other camera apps.');
    }
    if (warn.length) lines.push(`<span style="color:#fdd663;">&#9888; ${warn.join('<br>&#9888; ')}</span>`);
    return lines.join('<br>');
  }

  function review(frames, res, meta) {
    return new Promise(async (resolve, reject) => {
      let imgs;
      try {
        imgs = await Promise.all(
          frames.map((u) => {
            const im = new Image();
            im.src = u;
            return im.decode().then(() => im);
          })
        );
      } catch (e) {
        reject(e);
        return;
      }
      replay = { imgs, pos: 0, speed: 1, paused: false };
      $('qa').innerHTML = describeQA(res.qa, meta);
      $('scrub').value = '0';
      $('bPlay').textContent = 'Pause';
      root.querySelectorAll('.spd').forEach((b) => b.classList.toggle('on', b.dataset.s === '1'));
      $('live').style.display = 'none';
      $('review').style.display = 'block';
      ui.mode = 'replay';
      say('Replaying your take. Was it good?');

      const done = (decision) => {
        ui.mode = 'live';
        replay = null;
        $('review').style.display = 'none';
        $('live').style.display = 'flex';
        resolve(decision);
      };
      $('bGood').onclick = () => done('good');
      $('bBad').onclick = () => done('bad');
      $('bStop').onclick = () => {
        S.stop = true;
        done('stop');
      };
      $('bPlay').onclick = () => {
        replay.paused = !replay.paused;
        $('bPlay').textContent = replay.paused ? 'Play' : 'Pause';
      };
      $('bRestart').onclick = () => {
        replay.pos = 0;
        replay.paused = false;
        $('bPlay').textContent = 'Pause';
      };
      root.querySelectorAll('.spd').forEach((b) => {
        b.onclick = () => {
          replay.speed = parseFloat(b.dataset.s);
          root.querySelectorAll('.spd').forEach((x) => x.classList.toggle('on', x === b));
        };
      });
      $('scrub').oninput = () => {
        replay.paused = true;
        $('bPlay').textContent = 'Play';
        replay.pos = parseFloat($('scrub').value);
      };
    });
  }

  function waitForStartButton() {
    return new Promise((resolve) => {
      $('live').innerHTML = '';
      const go = document.createElement('button');
      go.className = 'aslb good';
      go.textContent = 'Start next take';
      go.onclick = () => resolve('go');
      const stop = document.createElement('button');
      stop.className = 'aslb';
      stop.textContent = 'Stop session';
      stop.onclick = () => {
        S.stop = true;
        resolve('stop');
      };
      $('live').append(go, stop);
      say('Ready when you are.');
    });
  }

  function showLiveControls() {
    $('live').innerHTML = '';
    const stop = document.createElement('button');
    stop.className = 'aslb';
    stop.textContent = 'Stop session';
    stop.onclick = () => {
      S.stop = true;
      stop.disabled = true;
      say('Stopping after the current step ...');
    };
    $('live').append(stop);
  }

  // ------------------------------------------------------- main
  try {
    $('hdrWord').textContent = CFG.wordDisplay;
    say('Starting camera - click Allow when the browser asks ...');
    const init = await call('asl.init', []);
    let saved = init.saved;
    $('hdrCount').textContent = `${saved} / ${CFG.target} saved`;
    if (saved >= CFG.target) {
      say(
        `Already have ${saved} takes for "${CFG.word}" (${CFG.person}). ` +
          'Raise TAKES in the config cell if you want more.',
        '#fdd663'
      );
      ui.mode = 'slate';
      ui.slate = 'Nothing to do - target already reached';
      requestAnimationFrame(render);
      return;
    }

    await startCamera();
    requestAnimationFrame(render);

    let takeId = 0;
    let first = true;
    while (!S.stop && saved < CFG.target) {
      takeId++;
      $('hdrCount').textContent = `take ${saved + 1} / ${CFG.target}  (${saved} saved)`;
      if (!first && !CFG.autoContinue) {
        await waitForStartButton();
        if (S.stop) break;
      }
      first = false;
      showLiveControls();
      await flashAndCountdown(saved + 1, saved);
      if (S.stop) break;
      const rec = await recordClip();
      if (!rec) break;
      $('live').innerHTML = '';
      const res = await sendAndCheck(takeId, rec.frames, rec.meta);
      const decision = await review(rec.frames, res, rec.meta);
      if (decision === 'good') {
        say('Saving ...');
        const r = await call('asl.save', [takeId]);
        saved = r.saved;
        note = `Saved ${r.file} (${saved}/${CFG.target}).  `;
        say(note, '#81c995');
      } else {
        await call('asl.discard', [takeId]);
        if (decision === 'bad') {
          note = 'Discarded - redoing.  ';
          say(note, '#fdd663');
        }
      }
    }

    stopCamera();
    const fin = await call('asl.finish', []);
    $('live').innerHTML = '';
    ui.mode = 'slate';
    ui.slate = `Session finished - ${fin.saved} takes saved`;
    $('hdrCount').textContent = `${fin.saved} / ${CFG.target} saved`;
    say(`Done. ${fin.saved} takes saved in ${fin.out_dir}. Run the next cell for a summary / zip download.`, '#81c995');
  } catch (err) {
    stopCamera();
    fail(err);
  }
})();
'''

register_callbacks()
display(Javascript(APP_JS.replace("__CFG__", json.dumps(ui_config()))))

In [ ]:
#@title 3. Summary + optional zip download { display-mode: "form" }
DOWNLOAD_ZIP = False #@param {type:"boolean"}
summarize()
if DOWNLOAD_ZIP:
    make_zip()